# Translating Wikipedia through LLMs
### Workshop · Arianit Dobroshi

Hands-on notebook: every cell can be run or copied as is.
Steps go from simplest to most complex, from what exists to what still
needs building.

| Step | What you do | Status |
|---|---|---|
| 0 | Pep talk, why, and Content Translation (the existing workflow) | |
| 1 | What a Wikipedia page is | ✅ |
| 2 | Meet and install the enwiki → sqwiki skill | ✅ |
| 3 | Run the skill by hand | ✅ |
| 4 | Verification by hand (Wikidata, templates) | ✅ |
| 5 | Improve the skill and commit | 🔧 |
| 6 | The automated harness | ✅ |
| 7 | Choosing models | ✅ / 🔧 |
| 8 | Sharing work through the queue | ✅ |
| 9 | Not built yet, and how to help | 🧭 |

✅ built today · 🔧 partial or built during the session · 🧭 not built yet

*Present as slides:* `jupyter nbconvert workshop-walkthrough.ipynb --to slides --post serve`

## Pep talk: learn by contributing

The best way to learn is to work on something real that other people use:
an open source project, a piece of software, Wikipedia, anything with a
public list of things to fix.

- You do not need to be an expert. A broken citation, a wrong term, a
  question nobody has answered yet: all of these count.
- Everything today is public: the skill, the harness, the queue. You can
  read it, run it, break it and fix it.
- **Human review by a Wikipedian is always mandatory.** Nothing produced
  today is published without a person who reads both languages checking
  the full text and its sources. The tools make drafts; people are
  responsible for what goes on sq.wikipedia.

**Direct motivation: grow sq.wikipedia.** Most of what readers look for
already exists in larger Wikipedias. Today's tools are a way to move some
of it into Albanian, with people checking the result.


## Tabs to open before starting

**The project**
- Skill: https://github.com/arianit/enwiki-sqwiki-translation
- Harness: https://github.com/arianit/wiki-translate-harness
- Architecture diagram: https://github.com/arianit/wiki-translate-harness/blob/main/docs/architecture.svg
- Changelog: https://github.com/arianit/wiki-translate-harness/blob/main/CHANGELOG.md
- Config reference: https://github.com/arianit/wiki-translate-harness/blob/main/config.example.yaml
- Benchmark (issue #1): https://github.com/arianit/wiki-translate-harness/issues/1
- Queue: https://github.com/arianit/wiki-translate-queue/blob/main/totranslate.txt
- Language-tech plan: https://github.com/arianit/albanian-language-tech-plan

**Wikipedia and Wikidata**
- Wikipedias by size: https://meta.wikimedia.org/wiki/List_of_Wikipedias
- sqwiki statistics: https://sq.wikipedia.org/wiki/Special:Statistics
- Articles every Wikipedia should have: https://meta.wikimedia.org/wiki/List_of_articles_every_Wikipedia_should_have
- Demo article (rendered): https://en.wikipedia.org/wiki/Caviidae
- Demo article (source): https://en.wikipedia.org/w/index.php?title=Caviidae&action=edit
- Its Wikidata item: https://www.wikidata.org/wiki/Special:ItemByTitle/enwiki/Caviidae
- Wikidata API sandbox: https://www.wikidata.org/wiki/Special:ApiSandbox
- enwiki `{{Cite web}}`: https://en.wikipedia.org/wiki/Template:Cite_web
- sqwiki `{{Cite web}}`: https://sq.wikipedia.org/wiki/Stampa:Cite_web
- Content Translation on sqwiki (log in first): https://sq.wikipedia.org/wiki/Special:ContentTranslation

**Policy**
- WP:LLM-assisted translation: https://en.wikipedia.org/wiki/Wikipedia:LLM-assisted_translation
- WP:Writing articles with LLMs: https://en.wikipedia.org/wiki/Wikipedia:Writing_articles_with_large_language_models
- Content Translation docs: https://www.mediawiki.org/wiki/Content_translation
- Wikimedia User-Agent policy: https://foundation.wikimedia.org/wiki/Policy:User-Agent_policy

**Tools and compute**
- Claude Code skills docs: https://docs.claude.com/en/docs/claude-code/skills
- OpenRouter models: https://openrouter.ai/models
- Experiential: https://github.com/experientiallabs/experiential
- OpenCode: https://github.com/sst/opencode
- Kosovo HPC & AI Competence Centre: https://hpc.kren-ks.eu/

## 0. Why, and where we start

- Wikipedia: 65M+ articles in 340+ languages. Albanian Wikipedia: ~100k.
- English and several other editions are deep and well sourced. For
  Albanian the content exists; it has not been moved over.
- Direct motivation: grow sq.wikipedia, one reviewed article at a time.
- Goal: 1,000, then 10,000, of the
  [Articles every Wikipedia should have](https://meta.wikimedia.org/wiki/List_of_articles_every_Wikipedia_should_have)
  in Albanian.
- LLMs make translation cheap. The bottleneck moves to **verification,
  review and coordination**.

🔗 **Open now:** [Wikipedias by size](https://meta.wikimedia.org/wiki/List_of_Wikipedias) · [sqwiki statistics](https://sq.wikipedia.org/wiki/Special:Statistics) · [Articles every Wikipedia should have](https://meta.wikimedia.org/wiki/List_of_articles_every_Wikipedia_should_have)

### Where we start: Wikimedia's Content Translation

> 📖 **Content Translation (CX)** is the translation tool built into Wikipedia. You pick an article in another language, it shows the source and your translation side by side, and a machine translation engine gives you a first draft to edit.

This is the existing workflow for sqwiki editors, so we start from it:

1. Open the tool on sq.wikipedia (log in first).
2. Choose a source article and language.
3. Edit the machine draft paragraph by paragraph, then publish.

What it does well: it is built in, needs no setup, and the editor stays in control. What we will look at: one article at a time, a draft from general machine translation, and links, templates and citation conventions that still need hand checks.


### Compared with Wikimedia's Content Translation

| | Content Translation (CX) | This pipeline |
|---|---|---|
| Draft from | machine translation (Google and others) | LLM following a versioned rule set |
| Links, categories | carried over automatically | verified via Wikidata, reported |
| Templates | carried over where mapped | checked on the target wiki, substituted or dropped |
| Human role | writes the translation from the MT draft | reviews the finished draft |
| Model choice, benchmarks | none | yes |

Policy is moving: English Wikipedia banned LLM-generated article text in
March 2026, with a narrow exception for translation under
[WP:LLM-assisted translation](https://en.wikipedia.org/wiki/Wikipedia:LLM-assisted_translation)
(translator must know both languages, check sources, remove errors).

🔗 **Open now:** [Content Translation on sqwiki](https://sq.wikipedia.org/wiki/Special:ContentTranslation) · [CX docs](https://www.mediawiki.org/wiki/Content_translation) · [WP:LLM-assisted translation](https://en.wikipedia.org/wiki/Wikipedia:LLM-assisted_translation) · [WP:Writing articles with LLMs](https://en.wikipedia.org/wiki/Wikipedia:Writing_articles_with_large_language_models)

### 🗣 Discussion

- Is 1,000 realistic in a year? Is 10,000? What limits it: compute, money, or reviewers?
- Which articles first: the 'every Wikipedia should have' list, Kosovo/Albania topics, or what readers search for and don't find?
- Who reviews? A translated article nobody reads before publishing is worse than no article.

## Setup

Set everything once here; later cells (Python and `%%bash`) read these.

Wikimedia asks every automated client to identify itself
([User-Agent policy](https://foundation.wikimedia.org/wiki/Policy:User-Agent_policy)).
Needs Python 3.10+, `requests`, `git`; for the run steps also
[Claude Code](https://docs.claude.com/en/docs/claude-code/overview) (`claude`).

The demo article is [Caviidae](https://en.wikipedia.org/wiki/Caviidae). **Pick
your own article instead:** not too long (a few paragraphs, two or three
citations), on a topic you know, with no sqwiki article yet if possible, so
the live run finishes in a few minutes. Put its exact enwiki title in
`DEMO_EN`.

In [ ]:
import os
import re
import requests
from pathlib import Path

settings = {
    "WIKIMEDIA_CONTACT": "your-name (you@example.com)",   # <- change
    "DEMO_EN": "Caviidae",      # <- or pick your own short enwiki article
    "WORK": str(Path.home() / "code"),   # where repos get cloned
    "ALLOW_QUEUE_WRITES": "no",     # "yes" only if you have write access to the shared queue
}
os.environ.update(settings)
os.environ["UA"] = f"wiki-translate-harness/0.1 ({settings['WIKIMEDIA_CONTACT']})"
os.environ["DEMO_SLUG"] = settings["DEMO_EN"].replace(" ", "_")
Path(settings["WORK"]).mkdir(parents=True, exist_ok=True)
Path("out").mkdir(exist_ok=True)

UA = os.environ["UA"]
for k in ["UA", "DEMO_EN", "WORK"]:
    print(f"{k:5} = {os.environ[k]}")

In [ ]:
%%bash
# What is installed?
for c in git python3 claude opencode gh; do
  printf "%-8s " "$c"; command -v "$c" >/dev/null && echo "ok" || echo "missing"
done

## 1. What a Wikipedia page is ✅

> 📖 **Wikitext** is the markup language Wikipedia pages are written in. A **template** is a reusable block written `{{like this}}`, for example an infobox or a citation. **Wikidata** is a shared database where each topic has one item linked to its article in every language.

Every article is stored as **wikitext**: prose mixed with links,
templates, citations and categories. Fetch one:

In [ ]:
def raw_wikitext(title: str, lang: str = "en") -> str:
    r = requests.get(
        f"https://{lang}.wikipedia.org/w/index.php",
        params={"title": title, "action": "raw"},
        headers={"User-Agent": UA},
        timeout=30,
    )
    r.raise_for_status()
    return r.text

text = raw_wikitext(os.environ["DEMO_EN"])
print(text[:2000])

🔗 **Open now:** [Rendered page](https://en.wikipedia.org/wiki/Caviidae) · [Edit source](https://en.wikipedia.org/w/index.php?title=Caviidae&action=edit) · [Raw wikitext](https://en.wikipedia.org/w/index.php?title=Caviidae&action=raw) (the default demo article)

In [ ]:
links = re.findall(r"\[\[([^\]|#]+)", text)
templates = re.findall(r"\{\{\s*([^|}\n]+)", text)
refs = re.findall(r"<ref[ >]", text)
cats = [l for l in links if l.startswith("Category:")]

print(f"wikilinks:  {len(links) - len(cats)}")
print(f"templates:  {len(templates)}  e.g. {sorted(set(t.strip() for t in templates))[:10]}")
print(f"references: {len(refs)}")
print(f"categories: {len(cats)}")

### Anatomy

```
== History ==                                      <- section heading
The town was founded in the 14th century and       <- prose
became a trading centre. According to the
historian [[John Smith]], it had about             <- wikilink
{{formatnum:15000}} inhabitants in 1850            <- template
<ref>{{cite book |last=Smith |title=...}}</ref>.   <- citation

{{Infobox settlement                               <- infobox = template
| name       = Example-on-Loire                    <-   with named parameters
| population = 12,458
}}

[[Category:Towns in France]]                       <- category
```

| Part | What happens to it |
|---|---|
| Prose | **translated** |
| Wikilinks | **verified**: does an sqwiki article exist, under which title? |
| Templates, infoboxes | **matched or substituted**: does sqwiki have it, with which parameters? |
| Citations, categories | **converted** to sqwiki conventions |

One article, four different jobs. Only one of them is translation.


### 🗣 Discussion

- Which of the four is hardest to get right? Which fails silently (renders fine, but wrong)?
- Many infoboxes pull their data from Wikidata. Should sqwiki infoboxes do the same, so nothing in them needs translating?
- Red links: leave them, use {{ill}}, or unlink? What does the sqwiki community prefer?

## 2. Meet the skill: enwiki → sqwiki ✅

> 📖 **Model**: a program trained on very large amounts of text that predicts and writes text. Claude, GPT and Qwen are models. On its own a model does not know sqwiki's conventions.
>
> 📖 **Assistant / Claude Code**: the program you talk to. Claude Code is the command line version, which can also run commands and read and write files.
>
> 📖 **Skill**: a folder of written instructions that the assistant loads when a task matches. The model stays the same; the skill tells it how we want the job done.

A skill is a folder with one required file, `SKILL.md`:

- **frontmatter** (`name`, `description`): tells the assistant *when* to use it
- **body**: tells it *how*
- optional `references/`: longer material, read only when needed

```
skill-name/
  SKILL.md        <- required
  references/     <- optional
```

The skill we use is
[`enwiki-sqwiki-translation`](https://github.com/arianit/enwiki-sqwiki-translation).
It holds the rules for turning English Wikipedia wikitext into sqwiki
wikitext: verified link and template names, source languages marked in
citations, ISO dates, Albanian number formatting, and transliteration of
names. It is a starting point: part of the exercise is running it and
fixing what breaks.


### 2.1 Install

Clone the repo and link every skill folder in it into `~/.claude/skills/`.
Then start a new Claude Code session and run `/skills` to check they are
listed. The harness's default config also expects `wikiterms` and `wikiqa`
there; if the public repo doesn't contain them yet, the config cell in 6.2
drops them and warns.

**Claude on the web:** zip the skill folder and upload it as a skill in
your Claude settings (skills must be enabled for your account).


In [ ]:
%%bash
cd "$WORK"
[ -d enwiki-sqwiki-translation ] || git clone -q https://github.com/arianit/enwiki-sqwiki-translation
cd enwiki-sqwiki-translation && git pull -q
mkdir -p ~/.claude/skills
find . -name SKILL.md -not -path "./.git/*" | while read f; do
  d=$(cd "$(dirname "$f")" && pwd)
  ln -sfn "$d" ~/.claude/skills/"$(basename "$d")"
  echo "linked $(basename "$d")"
done
ls -l ~/.claude/skills/

🔗 **Open now:** [Skill repo](https://github.com/arianit/enwiki-sqwiki-translation) · [Claude Code skills docs](https://docs.claude.com/en/docs/claude-code/skills)

### 2.2 Read it

A skill is plain text. Read what it tells the model to do.


In [ ]:
%%bash
ls ~/.claude/skills/enwiki-sqwiki-translation/
head -60 ~/.claude/skills/enwiki-sqwiki-translation/SKILL.md 2>/dev/null || echo "SKILL.md not found there; check where 2.1 linked it"

### 2.3 First test

**Interactive** (best for the live demo, the room sees each step):

```bash
claude
```
then type `/skills` to confirm the skill is listed, and paste:

```text
Translate this English Wikipedia article into Albanian for sq.wikipedia:
https://en.wikipedia.org/wiki/<DEMO_EN>
Give me the .wiki output and the verification report.
```

**Non-interactive** (the cell below): same request with `claude -p`,
output saved to `out/`. `--allowedTools` lets it call `curl` and write
files without stopping to ask. If it still stops on a permission, add the
tool it names to the list.

Check: did the skill trigger? Source language marked in the citations?
Numbers converted to `12.458`?

In [ ]:
%%bash
claude -p "Use the enwiki-sqwiki-translation skill. Translate https://en.wikipedia.org/wiki/$DEMO_SLUG into Albanian for sq.wikipedia. Write the wikitext to out/$DEMO_SLUG.v1.wiki and the verification report to out/$DEMO_SLUG.v1.report.md. Then print the report." \
  --allowedTools "Bash(curl:*)" "Read" "Write"

### 🗣 Discussion

- What belongs in a skill (judgment, conventions) and what belongs in code (lookups, fixes that must always happen)?
- One skill per source language, or one generic skill with a section per language? (Today there is one, for English.)
- Who maintains the terminology? Linguists editing a markdown file in git: realistic, or does it need a friendlier interface?

## 3. Run the skill by hand ✅

> 📖 **Prompt**: the message you send to the model. **Claude on the web** is the chat at claude.ai. **Claude Code** is the command line version that can fetch pages and write files, so it can check more.

**Claude on the web** (claude.ai, skill uploaded as zip): paste this with
the raw wikitext.

```text
Use the enwiki-sqwiki-translation skill on the article below.
Work section by section. Verify every wikilink and template before using it.
At the end give me: (1) the full .wiki file, (2) the verification report.

<paste raw wikitext here>
```

**Claude Code**: 2.3 already ran the skill from a URL. The cell below runs
it again from a pasted file, so the room can compare.

In [ ]:
%%bash
# Same article translated from a pasted file instead of a URL
# (what you'd do on the web, or offline)
curl -s -A "$UA" "https://en.wikipedia.org/w/index.php?title=$DEMO_SLUG&action=raw" > "out/$DEMO_SLUG.en.wiki"
wc -l "out/$DEMO_SLUG.en.wiki"
claude -p "Use the enwiki-sqwiki-translation skill on the English wikitext in out/$DEMO_SLUG.en.wiki. Do not fetch anything; work only from the file and your own knowledge, and mark every link and template you could not verify. Write out/$DEMO_SLUG.offline.wiki and out/$DEMO_SLUG.offline.report.md." \
  --allowedTools "Read" "Write"

Compare the report from the online run (2.3) with the offline one: the
offline run is roughly what Claude on the web can do without shell access.

In [ ]:
%%bash
ls -la out/
echo "--- unverified items, online vs offline ---"
grep -ci "not verified\|unverified\|not found" out/*.report.md 2>/dev/null || echo "(no reports yet)"

### Claude Code vs. Claude on the web

| | Claude Code (command line) | Claude on the web |
|---|---|---|
| Fetch source | `curl ... action=raw` | paste, or single page fetch |
| Link checks | 50 titles per Wikidata call, exact JSON | one search or page fetch at a time |
| Template parameters | reads live template source | approximate, from rendered pages |
| Citation language from URL | can visit each URL | rarely practical |
| Cache between sessions | files on disk | none |
| Long articles | can split and loop | manual copy per section |

Same skill, same rules. The web can browse, but coarsely: a weaker mode of
the same skill, not an equivalent.

### 🗣 Discussion

- Is Claude on the web good enough for an occasional editor translating one article a month?
- CX puts the human in the writing seat; this puts them in the reviewing seat. Which produces better articles? Which will the sqwiki community accept?
- What should sqwiki's own rule on LLM-assisted translation be? Is there one today?
- Attribution: the source oldid in the edit summary and {{Përkthyer nga}} on the talk page. Enough, or should LLM use also be declared?

## 4. Verification by hand ✅

> 📖 **API**: a web address a program can ask questions, getting data back instead of a page. A **sitelink** is Wikidata's record of which article, in which language, covers a topic. **TemplateData** is a template's own list of the parameters it accepts.

What the skill asks the model to do, and what the harness automates later.

### 4.1 Which linked articles exist on sqwiki (50 per call)

In [ ]:
def sq_sitelinks(titles: list[str], source: str = "enwiki") -> dict:
    """Map each source title to its sqwiki title, or None."""
    out = {}
    for i in range(0, len(titles), 50):
        batch = titles[i : i + 50]
        r = requests.get(
            "https://www.wikidata.org/w/api.php",
            params={
                "action": "wbgetentities",
                "sites": source,
                "titles": "|".join(batch),
                "props": "sitelinks",
                "sitefilter": f"{source}|sqwiki",
                "format": "json",
            },
            headers={"User-Agent": UA},
            timeout=30,
        )
        r.raise_for_status()
        for ent in r.json().get("entities", {}).values():
            src = ent.get("sitelinks", {}).get(source, {}).get("title")
            sq = ent.get("sitelinks", {}).get("sqwiki", {}).get("title")
            if src:
                out[src] = sq
        for t in batch:
            out.setdefault(t, None)
    return out

article_links = sorted({l.strip() for l in links if ":" not in l})
result = sq_sitelinks(article_links)
for en, sq in result.items():
    print(f"{'OK ' if sq else '-- '} {en:40} -> {sq or '(no sqwiki article)'}")

### 4.2 Notable enough for an `{{ill}}` link?

Rough proxy used by the harness: how many Wikipedias have an article on it.

In [ ]:
def sitelink_count(title: str, source: str = "enwiki") -> int:
    r = requests.get(
        "https://www.wikidata.org/w/api.php",
        params={"action": "wbgetentities", "sites": source, "titles": title,
                "props": "sitelinks", "format": "json"},
        headers={"User-Agent": UA}, timeout=30,
    )
    ents = r.json().get("entities", {})
    return max((len(e.get("sitelinks", {})) for e in ents.values()), default=0)

print(sitelink_count("League of Prizren"))

### 4.3 Which templates exist on sqwiki (namespace `Stampa`)

In [ ]:
def sq_templates_exist(names: list[str]) -> dict:
    titles = [f"Stampa:{n}" for n in names]
    r = requests.get(
        "https://sq.wikipedia.org/w/api.php",
        params={"action": "query", "titles": "|".join(titles), "format": "json"},
        headers={"User-Agent": UA}, timeout=30,
    )
    pages = r.json()["query"]["pages"].values()
    return {p["title"]: "missing" not in p for p in pages}

print(sq_templates_exist(["Cite book", "Cite web", "Cite journal", "Ill"]))

### 4.4 Which parameters a template really accepts

Uses TemplateData where it exists. Empty list: read the template's
documentation page instead. Use this to check the skill's template tables.

In [ ]:
def template_params(template: str, lang: str = "en", ns: str = "Template") -> list:
    r = requests.get(
        f"https://{lang}.wikipedia.org/w/api.php",
        params={"action": "templatedata", "titles": f"{ns}:{template}", "format": "json"},
        headers={"User-Agent": UA}, timeout=30,
    )
    for p in r.json().get("pages", {}).values():
        return sorted(p.get("params", {}).keys())
    return []

print("enwiki {{Cite web}}:", template_params("Cite web"))
print("sqwiki {{Cite web}}:", template_params("Cite web", lang="sq", ns="Stampa"))

🔗 **Open now:** [Wikidata item for the demo article](https://www.wikidata.org/wiki/Special:ItemByTitle/enwiki/Caviidae) · [Wikidata API sandbox](https://www.wikidata.org/wiki/Special:ApiSandbox) · [Same query as a URL](https://www.wikidata.org/w/api.php?action=wbgetentities&sites=enwiki&titles=Caviidae&props=sitelinks&sitefilter=enwiki|sqwiki&format=json) · [enwiki {{Cite web}}](https://en.wikipedia.org/wiki/Template:Cite_web) · [sqwiki {{Cite web}}](https://sq.wikipedia.org/wiki/Stampa:Cite_web)

### 🗣 Discussion

- Wikidata sitelinks as the source of truth for links: where does that break? (Disambiguation pages, articles merged differently on each wiki.)
- Is 'number of Wikipedias with an article' a fair test of notability for an {{ill}} link? What would be better?
- sqwiki infoboxes that kept English parameter names: a problem to fix, or a feature that makes translation easier?

## 5. Improve the skill and commit 🔧

> 📖 **Git** records every change to files so you can compare and undo. A **commit** is a saved snapshot with a message. A **fork** is your own copy of someone's project on GitHub. An **issue** is a note on GitHub that reports a problem or an idea.

Fix what went wrong in the skill, run the same article again, compare.

In [ ]:
%%bash
# The skill is a git clone, linked into ~/.claude/skills/, so edits take effect at once
cd "$WORK/enwiki-sqwiki-translation"
git checkout -q -b workshop-fixes 2>/dev/null || git checkout -q workshop-fixes
git status -sb && git log --oneline | head -5

Edit `SKILL.md` or the reference files in `$WORK/enwiki-sqwiki-translation`
(live, with the room), then run the cells below: translate again as v2 and
compare with v1 from 2.3. No reinstall is needed.


In [ ]:
%%bash
cd "$WORK/enwiki-sqwiki-translation" && git diff --stat && git diff | head -60

In [ ]:
%%bash
claude -p "Use the enwiki-sqwiki-translation skill. Translate https://en.wikipedia.org/wiki/$DEMO_SLUG into Albanian for sq.wikipedia. Write the wikitext to out/$DEMO_SLUG.v2.wiki and the verification report to out/$DEMO_SLUG.v2.report.md." \
  --allowedTools "Bash(curl:*)" "Read" "Write"
diff -u "out/$DEMO_SLUG.v1.wiki" "out/$DEMO_SLUG.v2.wiki" | head -80

In [ ]:
%%bash
cd "$WORK/enwiki-sqwiki-translation"
git -c user.name="workshop" -c user.email="workshop@example.com" commit -qam "Improve the skill (workshop)" || echo "(nothing changed yet)"
git log --oneline | head -5

Send it upstream through github.com:

1. **Fork** the repo: <https://github.com/arianit/enwiki-sqwiki-translation/fork>
2. Push your `workshop-fixes` branch to your fork and open a **pull request**, or
3. If you only found a problem, open an **issue**:
   <https://github.com/arianit/enwiki-sqwiki-translation/issues/new>

Suggested text for the issue:

```text
Tested on en:<article>.
What I tried:
What happened:
What I expected:
```


### 🗣 Discussion

- How do we review a change to a prompt? A code change has tests; what's the equivalent for a skill?
- Should every skill change be re-run against a fixed set of articles before merging?

## 6. The automated harness ✅

> 📖 **Harness**: a program that wraps a model in a repeatable process. It fetches the article, splits it into **chunks**, checks facts, calls the model, validates the answer and retries. An **engine** is the model service it calls. The **cache** remembers finished chunks so a rerun costs nothing.

[`wiki-translate-harness`](https://github.com/arianit/wiki-translate-harness)
does steps 3 and 4 for you, per chunk: fetch, split, verify, translate,
validate, repair, cache, report. It holds **no translation rules**; it
loads the skill.

```
fetch → split → cache? → prompt (skill + verified facts) → engine
      → validate → repair or flag for review → assemble → post-process → .wiki + report
```

Engines: Claude Code CLI (default, no API key), OpenRouter, any local
OpenAI-compatible server, Experiential, OpenCode Go.

🔗 **Open now:** [Harness repo](https://github.com/arianit/wiki-translate-harness) · [Architecture diagram](https://github.com/arianit/wiki-translate-harness/blob/main/docs/architecture.svg) · [config.example.yaml](https://github.com/arianit/wiki-translate-harness/blob/main/config.example.yaml)

### How the harness works

The pipeline, per article:

![Translation pipeline in three stages: prepare, per chunk, whole article](../docs/architecture.svg)

1. **Fetch and split** the article into chunks.
2. **Verify facts** on Wikidata and the target wiki: link targets, template names, infobox parameters.
3. **Translate each chunk** (several in parallel). Chunks already in the cache are reused.
4. **Validate** each chunk. On a defect the model is asked to repair it. If repair fails, the article fails and is not saved.
5. **Assemble** the article, apply fixes that need no model, validate again (including by rendering it through sqwiki's parse API), and write the `.wiki` file and a report.

Key features:

| Feature | What it means |
|---|---|
| No prompts of its own | The skill is loaded as the system prompt, from a pinned git revision so edits mid-run don't change a batch ([how the skill is used](https://github.com/arianit/wiki-translate-harness#how-the-skill-is-used)) |
| Facts checked in code | Lookups happen in the harness and reach the model as plain data ([fact verification](https://github.com/arianit/wiki-translate-harness#fact-verification)) |
| Validate and repair | Broken chunks are repaired or withheld; the model never has the last word |
| Fixes without a model | Citation language, citation parameter names, short footnotes ([deterministic fixes](https://github.com/arianit/wiki-translate-harness#deterministic-fixes)) |
| Cache and resume | Finished sections are remembered; rerunning a stopped job costs nothing for them |
| Model tiers | A cheaper draft model, a stronger one for infoboxes and tables, and an optional whole-article review pass ([model tiers](https://github.com/arianit/wiki-translate-harness#model-tiers)) |
| Five engines | Claude Code, OpenRouter, local server, Experiential, OpenCode Go, with a fallback when credits run out ([engines](https://github.com/arianit/wiki-translate-harness#engines)) |
| Queue and benchmark modes | Share work between people, compare models ([queue](https://github.com/arianit/wiki-translate-harness#queue-mode), [benchmark](https://github.com/arianit/wiki-translate-harness#benchmark-mode)) |
| Reports | Every run lists what was checked and what a reviewer should look at |
| Never publishes | Output is a file for a person to review and paste. Human Wikipedian review is always mandatory |

Deeper explanation: the harness [README](https://github.com/arianit/wiki-translate-harness#readme) and the [architecture diagram](https://github.com/arianit/wiki-translate-harness/blob/main/docs/architecture.svg).


### 6.1 Install

Clones into `$WORK`, makes a virtual environment, installs the harness.
Takes a minute or two: run it before the session.

In [ ]:
%%bash
cd "$WORK"
[ -d wiki-translate-harness ] || git clone -q https://github.com/arianit/wiki-translate-harness
cd wiki-translate-harness && git pull -q
[ -d .venv ] || python3 -m venv .venv
.venv/bin/pip install -q -e .
.venv/bin/wiki-translation-harness --help | head -40

### 6.2 Config files for the workshop

Two configs from `config.example.yaml`:

- `config.workshop.yaml`: default engine (Claude Code CLI), output to
  `./output` instead of the shared queue repo
- `config.openrouter.yaml`: OpenRouter engine, needed for benchmark mode


In [ ]:
H = Path(os.environ["WORK"]) / "wiki-translate-harness"
base = (H / "config.example.yaml").read_text()

def setkey(text, key, value):
    new, n = re.subn(rf"^{key}:.*$", f"{key}: {value}", text, flags=re.M)
    assert n == 1, f"key {key!r} not found once in config.example.yaml"
    return new

def skill_block(names):
    found = [n for n in names if (Path.home() / ".claude/skills" / n / "SKILL.md").exists()]
    for n in set(names) - set(found):
        print(f"warning: ~/.claude/skills/{n} not found, left out of skill_path")
    return "skill_path:\n" + "".join(f"  - ~/.claude/skills/{n}\n" for n in found)

def set_skills(text, names):
    new, n = re.subn(r"^skill_path:\n(?:  - .*\n)+", skill_block(names), text, flags=re.M)
    assert n == 1, "skill_path block not found"
    return new

qa = Path.home() / ".claude/skills/wikiqa/SKILL.md"
if not qa.exists():
    print("warning: ~/.claude/skills/wikiqa not found; repair calls may fail. "
          "Point qa_skill_path at your QA skill in the config files below.")

base = setkey(base, "wikimedia_contact", f'"{os.environ["WIKIMEDIA_CONTACT"]}"')
base = setkey(base, "output_dir", "output")
base = set_skills(base, ["enwiki-sqwiki-translation", "wikiterms"])
(H / "config.workshop.yaml").write_text(base)

(H / "config.openrouter.yaml").write_text(setkey(base, "provider", "openrouter"))

for f in ["config.workshop.yaml", "config.openrouter.yaml"]:
    print("wrote", H / f)

### 6.3 Translate one article

Default engine: Claude Code CLI (`claude -p` under your login, no API key).
Watch the log in a second terminal:
`tail -f $WORK/wiki-translate-harness/logs/run.log`

In [ ]:
%%bash
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN"

In [ ]:
%%bash
# What came out
cd "$WORK/wiki-translate-harness"
if ls output/*.report.md >/dev/null 2>&1; then
  ls -lt output/ | head
  LATEST=$(ls -t output/*.report.md | head -1)
  echo "=== $LATEST"; cat "$LATEST"
  echo "=== stats"; cat stats.json
else
  echo "no output yet; check logs/errors.log"; tail -20 logs/errors.log 2>/dev/null || true
fi

### Run it from a terminal

The same thing outside the notebook, with your own article:

```bash
cd ~/code/wiki-translate-harness
source .venv/bin/activate
export WIKIMEDIA_CONTACT="your-name (you@example.com)"   # or set wikimedia_contact in config.yaml

wiki-translation-harness --title "Caviidae"
```

The title must match the English Wikipedia page exactly, including capital
and lowercase letters: "Breast cancer" has a lowercase "c", so
"Breast Cancer" will not work. Passing the full URL also works:

```bash
wiki-translation-harness --title "https://en.wikipedia.org/wiki/Breast_cancer"
```

The result is `<Title>.wiki` and `<Title>.report.md` in the output folder.
Add `--config config.workshop.yaml` to use the workshop settings from 6.2
(output goes to `./output`). A person reviews the result before anything is
published.


### 6.4 Other ways to feed it

```bash
wiki-translation-harness --titles articles.txt           # one title per line, lang:Title allowed
wiki-translation-harness --category "fr:Ville du Kosovo"  # a whole category
wiki-translation-harness --file article.wiki              # local wikitext
wiki-translation-harness --directory raw_articles/        # folder of .wiki files
wiki-translation-harness --title "https://sr.wikipedia.org/wiki/Призрен"   # full URL, any wiki
```

Same command again resumes from cache (free). `--force` retranslates.

In [ ]:
%%bash
# Batch from a file
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
printf "en:%s\n" "$DEMO_EN" > articles.txt
cat articles.txt
wiki-translation-harness --config config.workshop.yaml --titles articles.txt

### 6.5 Engines

All five take the same flags; only `--provider` (and its key) changes.
Run whichever you have access to.

In [ ]:
%%bash
# OpenRouter (pay per token): https://openrouter.ai/models
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
export OPENROUTER_API_KEY="${OPENROUTER_API_KEY:-sk-or-...}"
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN" --force \
  --provider openrouter --model deepseek/deepseek-v3.2

In [ ]:
%%bash
# Local model: start a llama.cpp server first, in another terminal:
#   llama-server -m qwen3-8b-q5_k_m.gguf --port 8080
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN" --force \
  --provider local --base-url http://127.0.0.1:8080/v1 --model qwen3-8b-q5-k-m

In [ ]:
%%bash
# Experiential gateway: https://github.com/experientiallabs/experiential
# Key from platform.experientiallabs.ai; model = an alias/id available on your account
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
export EXPLABS_API_KEY="${EXPLABS_API_KEY:-xpl_...}"
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN" --force \
  --provider experiential --model "<model id on your account>"

In [ ]:
%%bash
# OpenCode Go: runs the `opencode` CLI under its own login (https://github.com/sst/opencode)
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN" --force \
  --provider opencode_go

### 6.6 Mixing models in one run

| Flag | Used for |
|---|---|
| `--model` | ordinary chunks |
| `--complex-model` (+ `--complex-provider`) | infoboxes, tables, dense references |
| `--review-model` (+ `--review-provider`) | re-reads the assembled article for meaning errors |
| `--fallback-provider` | switch engine if credits run out |

In [ ]:
%%bash
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN" --force \
  --model claude-haiku-4-5-20251001 \
  --complex-model claude-sonnet-5 \
  --review-model claude-sonnet-5

### How it evolved

- **v0.1**: OpenRouter only, skill loading, cache, Wikidata verification,
  deterministic fixes (citation language, CS1 parameter names, footnote
  dedup), reports, benchmark mode.
- **v0.2**: blind judge-model evaluation.
- **Since**: skill split so the QA checklist loads only on repair calls
  (~29% fewer prompt tokens per ordinary call); Claude Code CLI engine;
  article-level terminology registry so later chunks reuse earlier
  renderings; Experiential and OpenCode Go engines; complex-chunk and
  review-model routing.

None of these changes touched the translation rules. Judgment lives in the
skill, mechanics in the harness.

🔗 **Open now:** [Changelog](https://github.com/arianit/wiki-translate-harness/blob/main/CHANGELOG.md) · [Commit history](https://github.com/arianit/wiki-translate-harness/commits/main/) · [Experiential](https://github.com/experientiallabs/experiential) · [OpenCode](https://github.com/sst/opencode) · [Kosovo HPC & AI Competence Centre](https://hpc.kren-ks.eu/)

### 🗣 Discussion

- Anything the harness fixes in code (CS1 parameter names, citation language) is something the model got wrong often enough to stop trusting it. What else belongs on that list?
- Running on someone else's API: pricing, rate limits and terms change. When is it worth running our own inference, and on what? (e.g. the Kosovo HPC & AI Competence Centre, hpc.kren-ks.eu)
- Wikimedia hosts open LLMs itself: [LiftWing](https://wikitech.wikimedia.org/wiki/Machine_Learning/LiftWing/Large_Language_Models) (draft page, details may change) serves `llm-qwen3-14b` (16K context) and `llm-qwen36-27b` (32K context) through an OpenAI-compatible API with no key. Limits for translation: 100 requests/hour for anonymous traffic, shared across models and far too low for batches (lifted only from Toolforge, Cloud VPS or for approved bots); `<think>` blocks to strip; about 35 tokens/second per request; a long skill prompt, the chunk and the output must fit in the context. The page says nothing about Albanian. Qwen3 lists Tosk Albanian among its supported languages, but there is no published Albanian translation benchmark. Thinking mode is on by default and probably not wanted for translation; whether the API can switch it off is untested. Judge it on natural Albanian, fidelity, terminology and wikitext integrity. Our untested guess: the 27B is usable but weaker than a Sonnet-class model, the 14B clearly worse. Could it serve as a cheap first pass on simple chunks while a stronger model handles complex and review steps? Would the community trust a pipeline that runs on Wikimedia infrastructure? Another route is to fine-tune an open model on reviewed translations (licensing is in the last discussion). Would a tuned mid-size model beat a general one of the same size?

## 7. Choosing models ✅ / 🔧

> 📖 A **token** is a piece of a word; models read, write and charge in tokens. A **benchmark** runs the same test on several models. A **judge model** is a model that scores the others' answers.

What matters here, and general leaderboards mostly don't measure:

- cost per article at 1,000 to 10,000 articles
- clean wikitext out, no chatter around it
- following the skill's rules
- **Albanian quality**, which few public benchmarks cover

Current setup: a cheaper model drafts (Haiku 4.5), a stronger one reviews
and repairs (Sonnet 5). Local models (Qwen3 via llama.cpp) as an even
cheaper first draft: 🔧 explored, not proven.

### Benchmark mode

Same article through several models, then a separate judge model scores
them blind (labelled A to D, random order). Uses `config.openrouter.yaml`
because the model ids are OpenRouter ids. Costs roughly a dollar.

In [ ]:
%%bash
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness benchmark --config config.openrouter.yaml --title "$DEMO_EN" \
  --model deepseek/deepseek-v3.2 \
  --model google/gemini-2.5-flash \
  --model mistralai/mistral-large \
  --model qwen/qwen3-235b-a22b \
  --judge-model anthropic/claude-sonnet-4.5

In [ ]:
%%bash
cd "$WORK/wiki-translate-harness"
cat quality/*/comparison.md 2>/dev/null | head -80 || true
cat quality/*/mapping.json 2>/dev/null || echo "(no benchmark results yet)"

Previous result ([issue #1](https://github.com/arianit/wiki-translate-harness/issues/1)):

| Model | Score | Cost | Speed |
|---|---|---|---|
| deepseek/deepseek-v3.2 | 9/10 | $0.062 | 220 tok/s |
| google/gemini-2.5-flash | 9/10 | $0.128 | 622 tok/s |
| mistralai/mistral-large | 8/10 | $0.542 | 128 tok/s |
| qwen/qwen3-235b-a22b | 6/10 | $0.183 | 177 tok/s |

One article, one judge. A starting point, not a verdict.

🔗 **Open now:** [Issue #1](https://github.com/arianit/wiki-translate-harness/issues/1) · [OpenRouter models and prices](https://openrouter.ai/models)

### 🗣 Discussion

- Can an LLM judge Albanian quality reliably? How would we find out? (Compare its scores with native-speaker editors on the same texts.)
- What goes in a fixed test set? A short biography, a heavily cited medical article, a long philosophy article, a battle with foreign names...
- Draft-then-review on every chunk, or escalate to the stronger model only when validation fails?
- Which models have you seen do well in Albanian, for any task?

## 8. Sharing work through the queue ✅

> 📖 A **queue** is a shared to-do list. Several people and machines take items from it, so nobody translates the same article twice.

[`wiki-translate-queue`](https://github.com/arianit/wiki-translate-queue):
a text file in git. Several people and machines work through one list
without translating the same article twice. Claims use
discard-and-redo instead of git merges, because git reports false
conflicts on adjacent short lines.

Line format in `totranslate.txt`: URL or title, then nothing (pending),
`CLAIMED <host> <time>`, `DONE` or `FAILED`.

🔗 **Open now:** [totranslate.txt](https://github.com/arianit/wiki-translate-queue/blob/main/totranslate.txt) · [Finished articles](https://github.com/arianit/wiki-translate-queue/tree/main/output) · [Queue history](https://github.com/arianit/wiki-translate-queue/commits/main/)

In [ ]:
%%bash
cd "$WORK"
[ -d wiki-translation-queue ] || git clone -q https://github.com/arianit/wiki-translate-queue wiki-translation-queue
cd wiki-translation-queue && git pull -q --ff-only
PENDING=$(grep -v '^#' totranslate.txt | grep -v '^[[:space:]]*$' | grep -v $'\t')
echo "pending: $(echo "$PENDING" | grep -c .)"
echo "claimed: $(grep -c $'\tCLAIMED' totranslate.txt)"
echo "done:    $(grep -c $'\tDONE' totranslate.txt)"
echo "failed:  $(grep -c $'\tFAILED' totranslate.txt)"
echo "--- next pending"; echo "$PENDING" | head -5

**The next two cells write to the shared queue** (commit and push). They do
nothing unless `ALLOW_QUEUE_WRITES` is `"yes"` in Setup. Without write
access, fork the queue repo and point `--queue-repo-dir` at your fork.

In [ ]:
%%bash
# Add an article to the queue
[ "$ALLOW_QUEUE_WRITES" = "yes" ] || { echo "skipped: ALLOW_QUEUE_WRITES is not yes"; exit 0; }
cd "$WORK/wiki-translation-queue"
echo "https://en.wikipedia.org/wiki/$DEMO_SLUG" >> totranslate.txt
git commit -qam "Queue en:$DEMO_EN" && git push -q && tail -3 totranslate.txt

In [ ]:
%%bash
# Drain it: claim, translate, mark DONE/FAILED, push. One article only.
[ "$ALLOW_QUEUE_WRITES" = "yes" ] || { echo "skipped: ALLOW_QUEUE_WRITES is not yes"; exit 0; }
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness queue --config config.workshop.yaml \
  --queue-repo-dir "$WORK/wiki-translation-queue" --max-articles 1

Or from your own script. Shown, not run: executing it claims a real line in
the shared queue.

```python
import sys
from pathlib import Path

repo_dir = Path("~/code/wiki-translation-queue").expanduser()
sys.path.insert(0, str(repo_dir))
import queue_lib

claimed = queue_lib.claim_next_pending(repo_dir)   # pull, claim, push
if claimed:
    line_no, url = claimed
    ok = translate(url)                             # your own function
    queue_lib.finish_line(repo_dir, line_no, url, "DONE" if ok else "FAILED")
```

### 🗣 Discussion

- Who decides what goes in the queue? Should the sqwiki community see and edit the list?
- Where does human review sit: before an article leaves the queue, or after it's on sqwiki?

## 9. Not built yet 🧭

> 📖 **Inference** is running a trained model to get an answer, which happens every time you translate. **Fine-tuning** is training an existing model a little more on your own examples, here reviewed translations, so it gets better at one job. **LoRA** is a cheap way to fine-tune by training a small add-on instead of the whole model. A **parallel corpus** is the same text in two languages, used to train and test translation.

| Item | Status | Where to start |
|---|---|---|
| Rules for other source languages (French, German, ...) in the skill | 🧭 | skill repo |
| Fixed evaluation set, judge checked against human editors | 🧭 | harness issues |
| Stronger model only when needed | 🧭 | harness issues |
| Local model good enough for a first draft | 🔧 | `--provider local` |
| Own inference | 🧭 | compute partners, e.g. [Kosovo HPC & AI Competence Centre](https://hpc.kren-ks.eu/) |
| Fine-tune a model on reviewed translations | 🧭 | needs reviewed pairs first |
| Tool-using agent vs. harness, measured | 🧭 | harness issues |
| Output as a parallel corpus | 🧭 | [albanian-language-tech-plan](https://github.com/arianit/albanian-language-tech-plan) |

### The long road: an Albanian model

1. Every reviewed article is a training pair: draft in, corrected text out.
2. Experiential can turn traffic into a fine-tuned model you own; a
   LoRA/adapter on a small open model is the other route.
3. Compute: rented GPUs, or a national resource like the
   [Kosovo HPC & AI Competence Centre](https://hpc.kren-ks.eu/)
   (EuroHPC EUROCC project, University of Prishtina and KREN).
4. The [Albanian Language Technology Development Plan](https://github.com/arianit/albanian-language-tech-plan)
   calls for 4 million parallel pairs and an Albanian LLM. This pipeline
   produces parallel en to sq text as a side effect.

None of this is built.

🔗 **Open now:** [Albanian language-tech plan](https://github.com/arianit/albanian-language-tech-plan/blob/main/albanian-language-tech-plan.md) · [Kosovo HPC & AI Competence Centre](https://hpc.kren-ks.eu/) · [Experiential](https://github.com/experientiallabs/experiential) · [Wikipedia licensing (CC BY-SA)](https://en.wikipedia.org/wiki/Wikipedia:Copyrights)

### 🗣 Discussion

- Licensing: translations of Wikipedia are CC BY-SA. A corpus built from them carries that licence, not CC0. Does that fit the language-tech plan's goals?
- Some model providers' terms restrict using their outputs to train other models. Does that limit fine-tuning on drafts they produced? Check each provider's terms before building on it.
- Who could host inference long term: FLOSSK, KREN, Wikimedia Foundation?
- Is fine-tuning worth it, or will general models get good enough at Albanian on their own?

## Contribute

Pick one 🧭 line, or anything from today's discussion, and **open an issue
before you leave**. Small is fine: one disputed term, one broken citation,
one "someone should test this".

| Repo | For |
|---|---|
| [enwiki-sqwiki-translation](https://github.com/arianit/enwiki-sqwiki-translation/issues) | terminology, new source languages |
| [wiki-translate-harness](https://github.com/arianit/wiki-translate-harness/issues) | engines, verification, fixes, local models, benchmarks |
| [wiki-translate-queue](https://github.com/arianit/wiki-translate-queue/issues) | which articles, coordination |
| [albanian-language-tech-plan](https://github.com/arianit/albanian-language-tech-plan/issues) | the bigger picture |

- **Techies:** a new engine, better verification, local models.
- **Linguists:** terminology, transliteration, source-language rules.
- **Wikipedians:** choose articles, review drafts, report what breaks on sqwiki.
- **Anyone with compute or hosting:** say so. Own inference is a standing ask.

**Before anything goes live on sq.wikipedia, human Wikipedian review is
always mandatory:** a person who reads both languages reviews the full text
and its sources. No exceptions, whatever the model, skill or score.

### How to open an issue on github.com

1. Open the repo's **Issues** page (links in the table above) and press **New issue**, or use a direct link:
   - [enwiki-sqwiki-translation](https://github.com/arianit/enwiki-sqwiki-translation/issues/new)
   - [wiki-translate-harness](https://github.com/arianit/wiki-translate-harness/issues/new)
   - [wiki-translate-queue](https://github.com/arianit/wiki-translate-queue/issues/new)
   - [albanian-language-tech-plan](https://github.com/arianit/albanian-language-tech-plan/issues/new)
2. Write a short title.
3. Paste this into the description and fill it in:

```text
What I tried:
What happened:
What I expected:
```

No GitHub account yet? It is free: <https://github.com/signup>.
